#widgets

In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("schemaBronze", "bronze")
dbutils.widgets.text("catalogo", "project_dev")
dbutils.widgets.text("storageName", "adlssmartprojectdev13")

In [0]:
container = dbutils.widgets.get("container")
schemaBronze = dbutils.widgets.get("schemaBronze")
catalogo = dbutils.widgets.get("catalogo")
storageName = dbutils.widgets.get("storageName")

# definir rutas
path_inicio = f"abfss://{container}@{storageName}.dfs.core.windows.net/raw_data_project/"
path_target = f"{catalogo}.{schemaBronze}."

In [0]:
# importar functions essentials
from pyspark.sql.functions import *
from pyspark.sql.types import *

## CUSTOMERS

In [0]:
customers_schema = StructType(fields=[StructField("customer_id", StringType(), False),
                                     StructField("customer_unique_id", StringType(), True),
                                     StructField("customer_zip_code_prefix", IntegerType(), True),
                                     StructField("customer_city", StringType(), True),
                                     StructField("customer_state", StringType(), True)
])

In [0]:
df_customers = spark.read.format("csv") \
    .option("header", True) \
    .schema(customers_schema) \
    .load(f"{path_inicio}olist_customers_dataset.csv")

## SELLERS

In [0]:
df_sellers = spark.read.format("csv") \
    .option("header", True) \
    .load(f"{path_inicio}olist_sellers_dataset.csv")

## Geolocation

In [0]:
# leer el csv
df_geolocation = spark.read.format("csv") \
    .option("header", True) \
    .load(f"{path_inicio}olist_geolocation_dataset.csv")

## cargar informacion

In [0]:
df_customers.write.mode("overwrite").insertInto(f"{path_target}customers")

In [0]:
df_sellers.write.mode("overwrite").insertInto(f"{path_target}sellers")

In [0]:
df_geolocation.write.mode("overwrite").insertInto(f"{path_target}geolocation")

## aplicar optimize y vacuum

In [0]:
%sql
---- aplicar optimizer a la tabla target
OPTIMIZE ${catalogo}.${esquema_target}.customers
ZORDER BY customer_id;

OPTIMIZE ${catalogo}.${esquema_target}.sellers
ZORDER BY seller_id;

OPTIMIZE ${catalogo}.${esquema_target}.geolocation
ZORDER BY geolocation_zip_code_prefix;

In [0]:
%sql
--- habilitar vacum dinamico
SET spark.databricks.delta.retentionDurationCheck.enabled = false;

In [0]:
%sql
---- solo tener archivos inferiores  24 horas
VACUUM ${catalogo}.${esquema_target}.customers RETAIN 24 HOURS ;

VACUUM ${catalogo}.${esquema_target}.sellers RETAIN 24 HOURS;

VACUUM ${catalogo}.${esquema_target}.geolocation RETAIN 24 HOURS;